# Production Batch Creation

This notebook simulates incoming production data
for drift monitoring, since i don't have real production data.


## Objective

- Load the original dataset
- Select unseen data (test set)
- Apply the same preprocessing
- Save it as a production batch


In [ ]:
# imports
import pandas as pd
import numpy as np
import joblib
from pathlib import Path


## Load Raw Dataset


In [ ]:
df = pd.read_csv("../data/raw/creditcard.csv")


## Recreate Train / Validation / Test Split

The test set will be treated as new production data.


In [ ]:
from sklearn.model_selection import train_test_split

np.random.seed(42)

X = df.drop(columns=["Class"])
y = df["Class"]

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42
)

X_test.shape


## Apply Preprocessing

Only the features used during scaler training are transformed.
PCA features are left unchanged.

In [ ]:
scaler = joblib.load("../models/standard_scaler.pkl")

scale_cols = ["Time", "Amount"]

X_test_scaled = X_test.copy()

X_test_scaled[scale_cols] = scaler.transform(X_test[scale_cols])

X_test_scaled.head()

## Save Production Batch


In [ ]:
output_path = Path("../data/production/batches")

# Create folder if it does not exist
output_path.mkdir(parents=True, exist_ok=True)

# Save production batch
pd.DataFrame(X_test_scaled, columns=X_test.columns).to_parquet(
    output_path / "batch_001.parquet"
)

## Summary

- Production batch successfully created
- Same preprocessing pipeline applied as training
- Batch saved to `data/production/batches/`
- Ready for drift detection
